# 📚 HAL Metadata Snapshot Exploration

This notebook explores the local parquet snapshot stored in `data/hal_metadata_snapshot/data/*.parquet` using **DuckDB** and **Pandas**.



In [7]:
import duckdb
import pandas as pd

# Adjust pandas display limits for readability
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', None)

DATA_PATH = "../data/hal_metadata_snapshot/data/*.parquet"

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.5.5


## 1. Schema & Total Count

Check the schema of the parquet files and the total number of paper records.

In [8]:
# Inspect schema and column types
schema_df = con.sql(f"DESCRIBE SELECT * FROM '{DATA_PATH}'").df()
display(schema_df)

# Total count of papers
total_count = con.sql(f"SELECT count(*) AS total_papers FROM '{DATA_PATH}'").fetchone()[0]
print(f"\nTotal Papers in Dataset: {total_count:,}")

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,version,SMALLINT,YES,None,None,None
2,url,VARCHAR,YES,None,None,None
3,pdf_url,VARCHAR,YES,None,None,None
4,title,VARCHAR,YES,None,None,None
5,titles,"STRUCT(lang VARCHAR, ""text"" VARCHAR)[]",YES,None,None,None
6,authors,VARCHAR[],YES,None,None,None
7,contributors,VARCHAR[],YES,None,None,None
8,abstract,VARCHAR,YES,None,None,None
9,abstracts,"STRUCT(lang VARCHAR, ""text"" VARCHAR)[]",YES,None,None,None



Total Papers in Dataset: 4,610,856


## 2. Duplicate ID Check

Check if there are any duplicate records sharing the same CSIC `id`.


In [9]:
# Check for duplicate arXiv IDs
dup_check_df = con.sql(f"""
    SELECT
        count(*) AS total_records,
        count(DISTINCT id) AS distinct_ids,
        count(*) - count(DISTINCT id) AS duplicate_count
    FROM '{DATA_PATH}'
""").df()

display(dup_check_df)

dup_count = dup_check_df["duplicate_count"].iloc[0]
if dup_count > 0:
    print(f"\n⚠️ Found {dup_count:,} duplicate arXiv ID records.")
    sample_dups = con.sql(f"""
        SELECT
            id,
            count(*) AS occurrences
        FROM '{DATA_PATH}'
        GROUP BY id
        HAVING count(*) > 1
        ORDER BY occurrences DESC
        LIMIT 10
    """).df()
    print("Sample duplicate IDs:")
    display(sample_dups)
else:
    print("\n✅ No duplicate arXiv IDs found. All records are unique.")


,total_records,distinct_ids,duplicate_count
0,4610856,4610856,0



✅ No duplicate arXiv IDs found. All records are unique.


## 3. Items types

In [10]:
# Unnest and aggregate distinct values of type_eurepo
eurepo_df = con.sql(f"""
    SELECT
        unnest(type_eurepo) AS type_eurepo,
        count(*) AS count
    FROM '{DATA_PATH}'
    GROUP BY type_eurepo
    ORDER BY count DESC
""").df()

display(eurepo_df)

,type_eurepo,count
0,info:eu-repo/semantics/article,2267962
1,info:eu-repo/semantics/conferenceObject,1173512
2,info:eu-repo/semantics/bookPart,414360
3,info:eu-repo/semantics/other,247216
4,info:eu-repo/semantics/doctoralThesis,190508
5,info:eu-repo/semantics/book,126762
6,info:eu-repo/semantics/preprint,95807
7,info:eu-repo/semantics/report,80672
8,info:eu-repo/semantics/patent,8867
9,info:eu-repo/semantics/lecture,5190


In [14]:
# Unnest and aggregate distinct values of domains
domains_df = con.sql(f"""
    SELECT
        domain,
        count(*) AS count
    FROM (
        SELECT unnest(domains) AS domain
        FROM '{DATA_PATH}'
    )
    GROUP BY domain
    ORDER BY count DESC
""").df()

print(f"Total distinct domains: {len(domains_df)}")
display(domains_df.iloc[:100])

Total distinct domains: 1267


,domain,count
0,[SHS]Humanities and Social Sciences,493408
1,[SDV]Life Sciences [q-bio],406896
2,[SHS.DROIT]Humanities and Social Sciences/Law,239184
3,[SHS.HIST]Humanities and Social Sciences/History,212297
4,[SDE]Environmental Sciences,152949
...,...,...
95,[SHS.ENVIR]Humanities and Social Sciences/Environmental studies,14257
96,[INFO.INFO-CL]Computer Science [cs]/Computation and Language [cs.CL],13994
97,[SPI.MECA.MEFL]Engineering Sciences [physics]/Mechanics [physics.med-ph]/Fluids mechanics [physics.class-ph],13866
98,[SPI.ACOU]Engineering Sciences [physics]/Acoustics [physics.class-ph],13642
